# Initial Setup

In [ ]:
pip install musdb

In [ ]:
import musdb
import librosa
import numpy as np
import matplotlib.pyplot as plt

# Suppress divide-by-zero warnings during manual NMF initialization
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
MUSDB_PATH = "/content/drive/MyDrive/musdb18"
TARGETS = ['drums', 'bass', 'vocals', 'other']

# Ranks based on acoustic variance TODO: Change later potentially?
RANKS = {'drums': 40, 'bass': 20, 'vocals': 60, 'other': 90}

STFT_PARAMS = {'n_fft': 2048, 'hop_length': 512, 'window': 'hann'}

def get_spectrogram(audio_signal):
    """Converts audio to a non-negative magnitude spectrogram."""
    if audio_signal.ndim > 1:
        audio_signal = np.mean(audio_signal, axis=1) # Averages L and R channels
    return np.abs(librosa.stft(audio_signal, **STFT_PARAMS))

In [ ]:
db = musdb.DB(root=MUSDB_PATH, subsets="train")

  # NMF with KL Divergence — Step-by-Step Explanation

  This function performs Non-Negative Matrix Factorization (NMF) from scratch, explicitly optimizing the Kullback-Leibler (KL) Divergence cost function. It decomposes an input magnitude spectrogram ($V$) into a Basis dictionary ($B$) and an Activation matrix ($W$) such that $V \approx BW$.
  1. **Algorithm:** It implements the Lee and Seung Multiplicative Update (MU) rules. This guarantees that all matrix elements remain strictly non-negative throughout the optimization process, avoiding the boundary issues of standard additive gradient descent.
  2. **Memory Optimization:** Instead of computing large intermediate matrices of ones ($\mathbf{1}$) as shown in theoretical equations, the code uses heavily vectorized NumPy row/column summations and broadcasting. This prevents memory overflow and accelerates execution.
  3. **Acoustic Alignment:** It uses KL divergence (rather than the default Frobenius norm) because its scale-invariant properties perfectly map to the physical behavior of sound, ensuring low-energy harmonics are preserved alongside high-energy transients.

In [ ]:
def nmf(V, K, n_iterations=100):
    """
    Manually factorizes V ≈ BW using KL divergence multiplicative updates.
    V: Magnitude spectrogram (Frequency x Time)
    K: Target rank (number of basis vectors)
    Returns: B (Basis matrix)
    """

    F, T = V.shape
    eps = 1e-6 # Avoid Floating Point 0 Division

    V = V / (V.max() + eps)

    # Step 1: We want to initialize B and W with random positive values
    np.random.seed(101)
    B = np.random.rand(F, K) + eps # Basis Matrix
    W = np.random.rand(K, T) + eps # Weight Matrix

    print(f"  We're optimizing matrix of shape {V.shape} into Rank {K}")

    for i in range(n_iterations):
      # --- Update B ---
      V_hat = B @ W + eps                                      # eps every time
      B_denom = np.sum(W, axis=1, keepdims=False)              # shape (K,)
      B = B * ((V / V_hat) @ W.T) / (B_denom[np.newaxis, :] + eps)
      B = np.maximum(B, eps)                                   # clamp

      # --- L1 Normalization ---
      b_norms = np.sum(B, axis=0) + eps

      # B is (F, K), divide by (1, K) row vector
      B = B / b_norms[np.newaxis, :]

      # W is (K, T), multiply by (K, 1) column vector to compensate
      W = W * b_norms[:, np.newaxis]

      # --- Update W ---
      V_hat = B @ W + eps                                      # recompute with new B
      W_denom = np.sum(B, axis=0, keepdims=False)              # shape (K,)
      W = W * (B.T @ (V / V_hat)) / (W_denom[:, np.newaxis] + eps)
      W = np.maximum(W, eps)                                   # clamp

    return B

Supervised Dictionary Training

In [ ]:
basis_matrices = {}

MAX_FRAMES_PER_TARGET = 25000
NUM_TRACKS_TO_SAMPLE = 40  # Sample widely across the dataset

for target in TARGETS:
    print(f"\nProcessing target: {target.upper()}")

    target_frames = []

    # 2. Iterate widely, but lightly
    for track in db.tracks[:NUM_TRACKS_TO_SAMPLE]:
        audio = track.targets[target].audio
        V_track = get_spectrogram(audio)

        # 3. Silent-Frame Filtering (-40 dB threshold)
        frame_energy = np.sum(V_track**2, axis=0)
        threshold = np.max(frame_energy) * (10**(-4.0))

        # Keep only frames with actual acoustic information
        V_active = V_track[:, frame_energy > threshold]

        if V_active.shape[1] > 0:
            target_frames.append(V_active)

    # Concatenate all active frames for this instrument
    V_target = np.hstack(target_frames)

    # 4. Shuffle and trim to budget
    if V_target.shape[1] > MAX_FRAMES_PER_TARGET:
        indices = np.random.choice(V_target.shape[1], MAX_FRAMES_PER_TARGET, replace=False)
        V_target = V_target[:, indices]

    print(f"  Active Frames extracted: {V_target.shape[1]}")

    # Run the optimized NMF
    B_target = nmf(V_target, RANKS[target], n_iterations=200)

    # Note: B_target is already L1 normalized by the optimized function
    basis_matrices[target] = B_target




Processing target: DRUMS
  Active Frames extracted: 25000
  We're optimizing matrix of shape (1025, 25000) into Rank 40

Processing target: BASS
  Active Frames extracted: 25000
  We're optimizing matrix of shape (1025, 25000) into Rank 20

Processing target: VOCALS
  Active Frames extracted: 25000
  We're optimizing matrix of shape (1025, 25000) into Rank 60

Processing target: OTHER
  Active Frames extracted: 25000
  We're optimizing matrix of shape (1025, 25000) into Rank 90


Apply a High Pass Filter

In [ ]:
print("\nApplying High-Pass Filters...")

SR, N_FFT = 44100, 2048

def hpf_bin(freq_hz):
    return int(round(freq_hz * N_FFT / SR))

# 1. Apply the filter (zeroing out low-frequency bins)
basis_matrices['vocals'][:hpf_bin(150), :] = 1e-6
basis_matrices['other'][:hpf_bin(60), :] = 1e-6    # most "other" content above 60 Hz

# 2. Re-normalize the modified dictionaries to maintain unit L1 norm
for target in ['vocals', 'other']:
    b_norms = np.sum(basis_matrices[target], axis=0) + 1e-10
    basis_matrices[target] = basis_matrices[target] / b_norms[np.newaxis, :]

print("Dictionary training and filtering complete.")


Applying High-Pass Filters...
Dictionary training and filtering complete.


In [ ]:
# Horizontally concatenate all normalized dictionaries into B_global
B_global = np.hstack([basis_matrices[t] for t in TARGETS])

print(f"Final Global Dictionary Shape (Frequency Bins x Total Rank): {B_global.shape}")

# Save the dictionary for use in the separation phase
np.save("B_global_musdb.npy", B_global)
print("Saved as 'B_global_musdb.npy'")
print(B_global)

Final Global Dictionary Shape (Frequency Bins x Total Rank): (1025, 210)
Saved as 'B_global_musdb.npy'
[[9.99783034e-07 1.11191574e-05 2.86347847e-05 ... 1.00002427e-06
  1.00000100e-06 1.00002702e-06]
 [1.77221588e-06 8.18275115e-06 2.91967112e-02 ... 1.00002427e-06
  1.00000100e-06 1.00002702e-06]
 [1.77703685e-06 6.31303273e-04 6.08011486e-02 ... 1.00002427e-06
  1.00000100e-06 1.00002702e-06]
 ...
 [9.99783034e-07 9.99250650e-07 1.00019116e-06 ... 9.99781227e-07
  9.99725094e-07 9.99810919e-07]
 [9.99783034e-07 9.99250650e-07 1.00019116e-06 ... 9.99781227e-07
  9.99725094e-07 9.99810919e-07]
 [9.99783034e-07 9.99250650e-07 1.00019116e-06 ... 9.99781227e-07
  9.99725094e-07 9.99810919e-07]]


# Second Step: Using Gradient Descent on fixed Basis Matrix

We now have a basis for all of our instruments. Since a each song separates uniquely depending on its content, we ideally assume that our basis captures all of the nuances of each instrument, so we fix this value. We then perform gradient descent to find our W matrix with a few changes:

1. **Temporal Cost Penalty:** NMF by itself considers each frame as an individual observation. However, music and real-world sound applications vary much slower as a function of time. For instruments such as the 'other', 'vocal' and 'bass', this applies heavily.
2. **L1 Penalty:** Drums are inherently transient, where each percussive hit only last for a short/temporary time. To enforce our weight matrix to exhibit this behavior, we impose a L1 penalty to increase sparse representations.

We define the Temporal Cost Penalty to be:

$$c_t(W) = \sum_j\frac{1}{\sigma^2}\sum_t (W_{k,t} - W_{k,t-1})^2$$

Such that $\sigma$ is the standard deviation of $W$. Using Lee's Multiplicative Methodology, we obtain the iterative scheme:

$$W_{new} = W_{old} \odot \frac{[\nabla_W C_{base}]^- + [\nabla_W J_{penalty}]^-}{[\nabla_W C_{base}]^+ + [\nabla_W J_{penalty}]^+}$$

Where J penalty corresponds to our sparsity minimization or our temporal continuity approach.

In [ ]:
B_global = np.load("B_global_musdb.npy")

In [ ]:
def compute_tc_gradients(W_smooth, T, eps=1e-10):
    """
    Calculates the penalty numerator and denominator matrices
    for variance-normalized temporal continuity.
    """
    # 1. Calculating W_prev and W_next
    W_prev = np.roll(W_smooth, 1, axis=1)
    W_next = np.roll(W_smooth, -1, axis=1)

    # Zero our Boundary Constraints
    W_prev[:, 0] = 0
    W_next[:, -1] = 0

    # 2. Calculate Numerator (N_k) and Denominator (D_k) of the penalty quotient
    N_k = np.sum((W_smooth - W_prev)**2, axis=1, keepdims=True)
    D_k = np.mean(W_smooth**2, axis=1, keepdims=True) + eps

    alpha = 1.0 / D_k
    beta = N_k / (T * (D_k**2))

    # 3. Compute gradient additions
    # Negative gradient terms (to be added to NMF Numerator)
    tc_num_addition = 2 * alpha * (W_prev + W_next) + 2 * beta * W_smooth

    # Positive gradient terms (to be added to NMF Denominator)
    tc_den_addition = 4 * alpha * W_smooth

    return tc_num_addition, tc_den_addition

In [ ]:
def separate_sources(V_mix, B_global, ranks, params, n_iterations=200):
    """
    Factorizes a mixed spectrogram using a fixed dictionary.
    """
    F, T = V_mix.shape
    K = B_global.shape[1]
    eps = 1e-10

    np.random.seed(67)
    W = np.random.rand(K, T) + eps

    print(f"Factorizing mixture with decoupling regularization terms")

    for i in range(n_iterations):
        V_hat = B_global @ W + eps

        # --- Base KL Divergence Components ---
        Numerator = B_global.T @ (V_mix / V_hat)
        Denominator = np.sum(B_global, axis=0)[:, np.newaxis] + np.zeros((1, T)) + eps

        # --- Apply Specific Regularization Per Instrument ---
        start_idx = 0
        for target in TARGETS: # ['drums', 'bass', 'vocals', 'other']
            rank = ranks[target]
            end_idx = start_idx + rank

            lam_l1 = params[target]['l1']
            lam_tc = params[target]['tc']

            # Apply L1 Sparsity (Adds directly to denominator)
            if lam_l1 > 0:
                Denominator[start_idx:end_idx, :] += lam_l1

            # Apply Temporal Continuity (Normalized)
            if lam_tc > 0:
                W_target = W[start_idx:end_idx, :]
                tc_num_add, tc_den_add = compute_tc_gradients(W_target, T, eps)
                Numerator[start_idx:end_idx, :] += lam_tc * tc_num_add
                Denominator[start_idx:end_idx, :] += lam_tc * tc_den_add

            start_idx = end_idx
        # --- Execute Multiplicative Update ---
        W = W * (Numerator / Denominator)

    return W

In [ ]:
import soundfile as sf
import IPython.display as ipd

# 1. Select a Test Track
# We select the last track in the database to test the separation.

test_track = db.tracks[-2]
print(f"Testing Separation on: {test_track.name}")

# Downmix the stereo mixture to mono for our NMF pipeline
mix_audio = np.mean(test_track.audio, axis=1)

# Compute the Complex STFT and the Magnitude Spectrogram (V_mix)
X_mix = librosa.stft(mix_audio, **STFT_PARAMS)
V_mix = np.abs(X_mix)

# Vocals: High sparsity (lots of silence), Low TC (jagged syllables)
# Other: Low sparsity (always playing), High TC (sustained chords)
REG_PARAMS = {
    'drums':  {'l1': 0.06,  'tc': 0.0},
    'bass':   {'l1': 0.005, 'tc': 0.3},
    'vocals': {'l1': 0.005, 'tc': 0.15},
    'other':  {'l1': 0.001, 'tc': 0.7}
}

# 2. Supervised Factorization (Calculate W)
W_global = separate_sources(
    V_mix=V_mix,
    B_global=B_global,
    ranks=RANKS,
    params=REG_PARAMS,
    n_iterations=200
)

# 3. Soft Masking (Wiener Filter) & ISTFT Reconstruction
start_idx = 0
separated_audio = {}

# Calculate the total estimated energy (Denominator for the Wiener Filter)
V_hat_total = (B_global @ W_global) + 1e-10

print("\nReconstructing Audio...")
for target in TARGETS:
    rank = RANKS[target]
    end_idx = start_idx + rank

    # Isolate the specific B and W matrices for this instrument
    B_i = B_global[:, start_idx:end_idx]
    W_i = W_global[start_idx:end_idx, :]

    # Calculate this instrument's estimated energy contribution
    V_i_hat = B_i @ W_i

    # Create the Soft Mask (Percentage of energy belonging to this instrument)
    Mask_i = V_i_hat / V_hat_total

    # Apply the mask to the original COMPLEX mixture to preserve phase
    X_i_separated = Mask_i * X_mix

    # Reconstruct the time-domain audio using Inverse STFT
    audio_reconstructed = librosa.istft(X_i_separated, **STFT_PARAMS)
    separated_audio[target] = audio_reconstructed

    # Move the index window forward for the next instrument in the concatenated matrix
    start_idx = end_idx

# 4. Save the Outputs
# print("\nSaving separated stems to disk...")
# for target, audio in separated_audio.items():
#     filename = f"{target}_separated.wav"
#     sf.write(filename, audio, test_track.rate)
#     print(f"  Saved: {filename}")

print("\nSeparation Complete!")

Testing Separation on: Young Griffo - Facade
Factorizing mixture with decoupling regularization terms

Reconstructing Audio...

Separation Complete!


## Experimental Design

The first graph depicts the separation of drums and bass

In [ ]:
# Extracting Matrices for Results
print("Extracting Activation Sub-Matrices for Analysis...")

# 1. Define the rank boundaries based on your dictionary
ranks = [RANKS['drums'], RANKS['bass'], RANKS['vocals'], RANKS['other']]
boundaries = [0] + list(np.cumsum(ranks))

# 2. Slice the Global Activation Matrix (W) into individual instruments
W_drums  = W_global[boundaries[0]:boundaries[1], :]
W_bass   = W_global[boundaries[1]:boundaries[2], :]
W_vocals = W_global[boundaries[2]:boundaries[3], :]
W_other  = W_global[boundaries[3]:boundaries[4], :]

# 3. Slice the Global Dictionary Matrix (B) to match
B_drums  = B_global[:, boundaries[0]:boundaries[1]]
B_bass   = B_global[:, boundaries[1]:boundaries[2]]
B_vocals = B_global[:, boundaries[2]:boundaries[3]]
B_other  = B_global[:, boundaries[3]:boundaries[4]]

# 4. Calculate the isolated magnitude estimates (V_hat = B * W)
V_drums_hat  = B_drums @ W_drums
V_bass_hat   = B_bass @ W_bass
V_vocals_hat = B_vocals @ W_vocals

print("Extraction Complete. Ready for Graphing.")

Extracting Activation Sub-Matrices for Analysis...
Extraction Complete. Ready for Graphing.


For Latex Purpose

In [ ]:
!apt-get update -qq
!apt-get install -y texlive-latex-recommended texlive-latex-extra texlive-fonts-recommended dvipng > /dev/null


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Failed to fetch https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu/dists/jammy/InRelease  Could not connect to ppa.launchpadcontent.net:443 (185.125.190.80), connection timed out
W: Failed to fetch https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu/dists/jammy/InRelease  Unable to connect to ppa.launchpadcontent.net:443:
W: Failed to fetch https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu/dists/jammy/InRelease  Unable to connect to ppa.launchpadcontent.net:443:
W: Some index files failed to download. They have been ignored, or old ones used instead.


In [ ]:
import matplotlib
matplotlib.use("pgf")
import matplotlib.pyplot as plt

plt.rcParams.update({
    "pgf.texsystem": "pdflatex",
    "font.family": "serif",  # Matches standard LaTeX fonts
    "text.usetex": True,
    "pgf.rcfonts": False,
})

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')

# Define how many seconds to skip, and how wide the viewing window should be
start_offset = 5.0
window_length = 10.0

# FIGURE 1: Proving Decoupled Regularization (Activations)
# Calculate the mean activation energy across all templates for a given time frame
drums_energy = np.mean(W_drums, axis=0)
bass_energy = np.mean(W_bass, axis=0)

# Normalize for visual comparison
drums_energy /= np.max(drums_energy)
bass_energy /= np.max(bass_energy)

# Convert frames to seconds for the X-axis
time_axis = librosa.frames_to_time(np.arange(W_global.shape[1]), sr=44100, hop_length=1024)

fig1, ax1 = plt.subplots(figsize=(10, 4), dpi=300)
ax1.plot(time_axis, drums_energy, label='Drums (L1 Sparsity Penalized)', color='darkred', alpha=0.8, linewidth=1.5)
ax1.plot(time_axis, bass_energy, label='Bass (Temporal Continuity Penalized)', color='navy', alpha=0.8, linewidth=2.5)

ax1.set_title('Figure 1: Effects of Decoupled Regularization on Temporal Activations (W)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Time (Seconds)', fontsize=12)
ax1.set_ylabel('Normalized Mean Activation Energy', fontsize=12)
ax1.legend(loc='upper right', frameon=True, shadow=True)

# Shift the X-axis limits using the offset variables
ax1.set_xlim([start_offset, min(start_offset + window_length, time_axis[-1])])

plt.tight_layout()
plt.savefig('activation_plt.pgf')
plt.show()

The next sequence of code cells gives us trade offs of cranking sparsity / temporal cost increasingly

In [ ]:
# EXTREME PARAMETERS: Designed to force a captivating SIR vs SAR tradeoff pattern
# Drum SIR skyrocket (perfect isolation), but Drum SAR plummet (heavy robotic artifacts).
REG_PARAMS = {
    'drums':  {'l1': 0.05, 'tc': 0.0},  # Extreme Sparsity
    'bass':   {'l1': 0.00, 'tc': 0.85}, # Extreme Smoothing
    'vocals': {'l1': 0.01, 'tc': 0.05},
    'other':  {'l1': 0.00, 'tc': 0.85}
}

In [ ]:
pip install mir_eval

In [ ]:
import mir_eval
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("Calculating SDR, SIR, and SAR metrics...")

targets = ['drums', 'bass', 'vocals', 'other']

# ==========================================
# Extract and downmix reference stems
# ==========================================
reference_stems = {}
for t in targets:
    # Extract stereo ground truth and average channels to mono
    reference_stems[t] = np.mean(test_track.targets[t].audio, axis=1)

# Convert dictionaries to aligned 2D numpy arrays (Shape: [n_sources, n_samples])
# Ensure both arrays are the exact same length to prevent broadcasting errors
min_len = min(len(reference_stems['vocals']), len(separated_audio['vocals']))

ref_matrix = np.array([reference_stems[t][:min_len] for t in targets])
est_matrix = np.array([separated_audio[t][:min_len] for t in targets])

# Compute BSS Eval Metrics
sdr, sir, sar, _ = mir_eval.separation.bss_eval_sources(ref_matrix, est_matrix, compute_permutation=False)

# Store results in a Pandas DataFrame for easy graphing
results_df = pd.DataFrame({
    'Instrument': targets * 3,
    'Metric': ['SDR']*4 + ['SIR']*4 + ['SAR']*4,
    'Score (dB)': np.concatenate([sdr, sir, sar])
})

# ==========================================
# GRAPHING: Metric Tradeoff Patterns
# ==========================================
plt.style.use('seaborn-v0_8-whitegrid')
fig, ax = plt.subplots(figsize=(10, 5), dpi=300)

# Create a grouped bar chart
sns.barplot(
    data=results_df,
    x='Instrument',
    y='Score (dB)',
    hue='Metric',
    palette=['#2c3e50', '#27ae60', '#e74c3c'],
    ax=ax
)

ax.set_title('Figure 2: BSS Evaluation Metrics (The SIR vs. SAR Tradeoff)', fontsize=14, fontweight='bold')
ax.set_xlabel('Target Source', fontsize=12)
ax.set_ylabel('Score in Decibels (dB)', fontsize=12)
ax.legend(title='BSS Metric', loc='upper left', frameon=True)

# Add exact text labels above the bars
for p in ax.patches:
    ax.annotate(f"{p.get_height():.1f}",
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=9, xytext=(0, 3),
                textcoords='offset points')

plt.tight_layout()
plt.savefig('bss_evaluation_metrics.png')
plt.show()

Calculating SDR, SIR, and SAR metrics...


In [ ]:
plt.savefig('bss_evaluation_metrics.pgf')
